In [1]:
%matplotlib widget
from easydynamics.analysis import Analysis


from easydynamics.sample import GaussianComponent
from easydynamics.sample import LorentzianComponent
from easydynamics.sample import PolynomialComponent
from easydynamics.sample import SampleModel
from easydynamics.sample import DeltaFunctionComponent

from easydynamics.experiment  import Experiment

from easydynamics.job import Job
from easydynamics.experiment  import Data

import matplotlib.pyplot as plt
import scipp as sc
import plopp as pp
plt.close()

In [2]:
MyJob= Job(name='Vanadium sample')

theory= SampleModel('Vanadium Model')
theory.add_component(GaussianComponent(width=0.5e-3,area=5e-5))
theory.add_component(LorentzianComponent(width=2e-3, area=2e-6))

MyJob.set_theory(theory)


experiment=Experiment('Vanadium Experiment')

background=SampleModel('Vanadium Background')
background.add_component(PolynomialComponent(coefficients=[1e-3]))


MyJob.set_background_model(background)
data=Data('Vanadium Data')
data.append(data.load_example_vanadium_data())

experiment.set_data(data)

MyJob.set_experiment(experiment)


analysis=Analysis('Vanadium Analysis')

analysis.set_background_model(background)

analysis.set_resolution_model(None)

# MyJob.set_analysis(analysis)

MyJob.generate_analysis_for_cuts()

In [3]:
MyJob._analysis[0].get_fit_parameters()

[<Parameter 'Gaussianarea': 5.000e-05 meV, bounds=[0.0:inf]>,
 <Parameter 'Gaussianwidth': 0.0005 meV, bounds=[0.0:inf]>,
 <Parameter 'Lorentzianarea': 2.000e-06 meV, bounds=[0.0:inf]>,
 <Parameter 'Lorentzianwidth': 0.0020 meV, bounds=[0.0:inf]>,
 <Parameter 'Polynomial_c0': 0.0010, bounds=[-inf:inf]>,
 <Parameter 'offset': 0.0000 meV, bounds=[-inf:inf]>]

In [4]:
MyJob.fit()

In [5]:
# for i in range(len(MyJob._analysis)):
#     fig = MyJob._analysis[i].plot_data_and_model(plot_individual_components=True)
#     fig.axes[0].set_xlim(-0.02, 0.02)
#     fig.axes[0].set_ylim(0, 0.07)




In [6]:
MyJob._experiment._data.data

<scipp.DataArray>
Dimensions: Sizes[Q:16, energy:1024, ]
Coordinates:
* Q                         float64           [1/Å]  (Q)  [0.5708, 0.7002, ..., 1.8965, 1.9361]
* energy                    float64            [meV]  (energy)  [-0.029678, -0.02962, ..., 0.02962, 0.029678]
Data:
                            float64  [dimensionless]  (Q, energy)  [0, 0, ..., 0, 0]  [1, 1, ..., 1, 1]

In [7]:

model = sc.zeros_like(MyJob._experiment._data.data)

# data['Q',i]
for i in range(len(MyJob._analysis)):
    model['Q',i].values = MyJob._analysis[i].calculate_theory(model['Q',i].coords['energy'].values)


data_and_fit = sc.DataGroup({'Data': MyJob._experiment._data.data,
                             'Fit': model})


INTENSITY_MIN_VANADIUM=0.0
INTENSITY_MAX_VANADIUM=0.06

ENERGY_MIN_VANADIUM = -0.02 * sc.Unit('meV')
ENERGY_MAX_VANADIUM = 0.02 * sc.Unit('meV')
pp.slicer(data_and_fit['energy',ENERGY_MIN_VANADIUM:ENERGY_MAX_VANADIUM],
          vmin=INTENSITY_MIN_VANADIUM,vmax=INTENSITY_MAX_VANADIUM,
               keep=['energy'],
     linestyle=         {'Data': 'none',    'Fit': '-'},
     marker=            {'Data': 'o',       'Fit':'none'},
     markerfacecolor=   {'Data': 'none',    'Fit':'red'},
     color=             {'Data': 'black',   'Fit':'red'})


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [8]:


MyJob.plot_data_and_model()


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [9]:
MyJob.analysis

[Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`,
 Analysis `MyAnalysis`]

In [10]:
pars=MyJob.analysis[6].get_parameters()
pars

[<Parameter 'temperature': -1.0000 K (fixed), bounds=[-inf:inf]>,
 <Parameter 'Gaussianarea': 5.295e-05 ± 4.046e-06 meV, bounds=[0.0:inf]>,
 <Parameter 'Gaussiancenter': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Gaussianwidth': 0.0004 ± 0.0000 meV, bounds=[0.0:inf]>,
 <Parameter 'Lorentzianarea': 3.807e-06 ± 3.768e-06 meV, bounds=[0.0:inf]>,
 <Parameter 'Lorentziancenter': 0.0000 meV (fixed), bounds=[-inf:inf]>,
 <Parameter 'Lorentzianwidth': 0.0005 ± 0.0005 meV, bounds=[0.0:inf]>,
 <Parameter 'temperature': -1.0000 K (fixed), bounds=[-inf:inf]>,
 <Parameter 'Polynomial_c0': 0.0001 ± 0.0000, bounds=[-inf:inf]>,
 <Parameter 'offset': -0.0003 ± 0.0000 meV, bounds=[-inf:inf]>]

In [11]:
pars=MyJob.get_parameters_as_data_group()
pars

DataGroup(sizes={'Q': 16}, keys=[
    temperature[0]: DataArray({'Q': 16}),
    Gaussianarea: DataArray({'Q': 16}),
    Gaussiancenter: DataArray({'Q': 16}),
    Gaussianwidth: DataArray({'Q': 16}),
    Lorentzianarea: DataArray({'Q': 16}),
    Lorentziancenter: DataArray({'Q': 16}),
    Lorentzianwidth: DataArray({'Q': 16}),
    temperature[1]: DataArray({'Q': 16}),
    Polynomial_c0: DataArray({'Q': 16}),
    offset: DataArray({'Q': 16}),
])

In [12]:
pp.plot(pars['Gaussianarea'])

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [13]:
pars2=MyJob.get_parameters_as_data_group_with_bounds()
pars2

DataGroup(sizes={'Q': 16}, keys=[
    temperature[0]: DataGroup(4, {'Q': 16}),
    Gaussianarea: DataGroup(4, {'Q': 16}),
    Gaussiancenter: DataGroup(4, {'Q': 16}),
    Gaussianwidth: DataGroup(4, {'Q': 16}),
    Lorentzianarea: DataGroup(4, {'Q': 16}),
    Lorentziancenter: DataGroup(4, {'Q': 16}),
    Lorentzianwidth: DataGroup(4, {'Q': 16}),
    temperature[1]: DataGroup(4, {'Q': 16}),
    Polynomial_c0: DataGroup(4, {'Q': 16}),
    offset: DataGroup(4, {'Q': 16}),
])

In [14]:
pars3=MyJob.get_parameters_as_data_group_with_T()
pars3

DataGroup(sizes={'Q': 16}, keys=[
    temperature[0]: DataGroup(4, {'Q': 16}),
    Gaussianarea: DataGroup(4, {'Q': 16}),
    Gaussiancenter: DataGroup(4, {'Q': 16}),
    Gaussianwidth: DataGroup(4, {'Q': 16}),
    Lorentzianarea: DataGroup(4, {'Q': 16}),
    Lorentziancenter: DataGroup(4, {'Q': 16}),
    Lorentzianwidth: DataGroup(4, {'Q': 16}),
    temperature[1]: DataGroup(4, {'Q': 16}),
    Polynomial_c0: DataGroup(4, {'Q': 16}),
    offset: DataGroup(4, {'Q': 16}),
])

In [29]:
datajob= Job(name='Sample')


dataexperiment=Experiment('Experiment')
newdata=Data('Data')
newdata.append(newdata.load_example_data())

dataexperiment.set_data(newdata)

datajob.set_experiment(dataexperiment)

datajob._experiment._data.data.plot()
Newtheory= SampleModel('Model')
Newtheory.add_component(DeltaFunctionComponent(area=0.01e-3))
Newtheory.add_component(LorentzianComponent(width=2e-3, area=5e-6,name='DataLorentzian'))

datajob.set_theory(Newtheory)

Newbackground=SampleModel('Background')
Newbackground.add_component(PolynomialComponent(coefficients=[1e-3]))


datajob.set_background_model(Newbackground)

datajob.generate_analysis_for_cuts()

datajob.use_fit_as_resolution(MyJob)




16
16


In [30]:
datajob.plot_data_and_model(intensity_min=0.0, intensity_max=0.015)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [31]:

# datajob._analysis[0].get_parameters()
datajob._analysis[0].get_fit_parameters()

[<Parameter 'DeltaFunctionarea': 1.000e-05 meV, bounds=[0.0:inf]>,
 <Parameter 'DataLorentzianarea': 5.000e-06 meV, bounds=[0.0:inf]>,
 <Parameter 'DataLorentzianwidth': 0.0020 meV, bounds=[0.0:inf]>,
 <Parameter 'Polynomial_c0': 0.0010, bounds=[-inf:inf]>,
 <Parameter 'offset': 0.0000 meV, bounds=[-inf:inf]>]

In [32]:
datajob.fit()

In [33]:
datajob.plot_data_and_model(intensity_min=0.0, intensity_max=0.015)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [20]:
fig=DataJob.analysis[-1].plot_data_and_model(plot_individual_components=True)

# # Extract data
# x, y, e = DataJob.analysis._experiment.extract_xye_data(DataJob.analysis._experiment._data)

# # Start plot
# fig = plt.figure(figsize=(10, 6))
# # plt.errorbar(x, y, yerr=e, label='Data', color='black', marker='o', linestyle='None', markerfacecolor='none')

# # Compute and plot fit
# fit_y = DataJob.analysis.calculate_theory(x)
# plt.plot(x, fit_y, label='Fit', color='red')



fig.axes[0].set_xlim(-0.02, 0.02)
fig.axes[0].set_ylim(0, 0.015)

NameError: name 'DataJob' is not defined

In [ ]:
DataJob.fit()
fig=DataJob.analysis[-1].plot_data_and_model(plot_individual_components=True)

fig.axes[0].set_xlim(-0.029, 0.029)
fig.axes[0].set_ylim(0, 0.015)


In [ ]:
DataJob.analysis[-1].get_fit_parameters()

In [ ]:
DataJob.analysis[-1].get_parameters()